# Bab 1 — Introduction to Clustering Experiments

Notebook pendamping Bab 1 Level 6.

Notebook ini mengikuti halaman bab di situs. Jalankan sel dari atas ke bawah.

## Learning Objectives

Setelah mempelajari bab ini, kamu mampu:

- Membedakan "menjalankan sebuah algoritma" dari "melakukan sebuah eksperimen".
- Menjelaskan mengapa eksperimen yang sistematis diperlukan pada clustering.
- Merumuskan tujuan eksperimen dan pertanyaan riset yang sesuai untuk masalah *unsupervised*.
- Mengenali variabel bebas (*independent*), variabel terikat (*dependent*), dan variabel terkontrol (*controlled*).
- Menentukan *baseline* dan mengukur kebisingan baseline.
- Menjelaskan alur eksperimen clustering dari pertanyaan sampai kesimpulan.
- Menjalankan satu eksperimen kecil yang terkontrol dengan Python, dan membaca hasilnya dengan hati-hati.
- Mengenali kesalahan umum dalam eksperimen clustering.

## Langkah 0: Pindah ke folder utama proyek

In [ ]:
import os
import sys
from pathlib import Path

if not Path("data").exists():
    os.chdir("../..")

# alat bantu pencatat eksperimen (examples/level-6/_experiment.py)
sys.path.insert(0, str(Path("examples/level-6").resolve()))

## Bagian A: Siapkan alat

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, silhouette_score
from sklearn.preprocessing import MinMaxScaler, StandardScaler

## Bagian B: Merumuskan pertanyaan dan variabel

In [ ]:
# Rancangan eksperimen ditulis lebih dulu, sebelum menjalankan apa pun
experiment = {
    "pertanyaan": "Apakah cara menyamakan skala mengubah pengelompokan pelanggan Mall Customers?",
    "variabel_bebas": "perlakuan skala fitur (4 kondisi)",
    "variabel_terkontrol": "algoritma K-Means, K = 5, n_init = 10, seed = 42, dua fitur yang sama",
    "variabel_terikat": "kesamaan pengelompokan dengan baseline (ARI), ukuran kelompok, Silhouette di ruang bersama",
    "baseline": "StandardScaler",
}
for key, value in experiment.items():
    print(f"{key:20s}: {value}")

## Bagian C: Dataset dan karakteristiknya

In [ ]:
df = pd.read_csv("data/mall_customers.csv")
features = ["Annual Income (k$)", "Spending Score (1-100)"]
X_raw = df[features].to_numpy(dtype=float)

print("Jumlah pelanggan :", len(df))
print("Fitur dipakai    :", features)
print(df[features].describe().loc[["min", "max", "mean", "std"]].round(2).to_string())

## Bagian D: Baseline dan kebisingan antar-seed

In [ ]:
SEED = 42
X_standard = StandardScaler().fit_transform(X_raw)   # kondisi baseline
baseline = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit_predict(X_standard)
print("Ukuran kelompok baseline:", sorted(np.bincount(baseline).tolist(), reverse=True))

# Kebisingan baseline: ulangi baseline dengan seed lain, lalu bandingkan dengan ARI
runs = [KMeans(n_clusters=5, n_init=10, random_state=seed).fit_predict(X_standard) for seed in range(10)]
ari_between_runs = [adjusted_rand_score(runs[0], run) for run in runs[1:]]
print("ARI baseline seed 0 vs seed 1..9, terkecil:", round(min(ari_between_runs), 4))
print("ARI baseline seed 0 vs seed 1..9, terbesar:", round(max(ari_between_runs), 4))

## Bagian E: Menjalankan empat kondisi

In [ ]:
X_dollars = X_raw.copy()
X_dollars[:, 0] *= 1000            # pendapatan dalam dolar, bukan ribu dolar

conditions = {
    "A. StandardScaler (baseline)": X_standard,
    "B. MinMaxScaler": MinMaxScaler().fit_transform(X_raw),
    "C. Tanpa skala (k$)": X_raw,
    "D. Tanpa skala (pendapatan dalam dolar)": X_dollars,
}

labels_by_condition = {}
for name, X_condition in conditions.items():
    labels_by_condition[name] = KMeans(n_clusters=5, n_init=10, random_state=SEED).fit_predict(X_condition)
    sizes = sorted(np.bincount(labels_by_condition[name]).tolist(), reverse=True)
    print(f"{name:42s} ukuran kelompok: {sizes}")

## Bagian F: Mengukur di ruang yang sama

In [ ]:
rows = []
for name, X_condition in conditions.items():
    labels = labels_by_condition[name]
    rows.append(
        {
            "Kondisi": name,
            "ARI vs baseline": adjusted_rand_score(baseline, labels),
            "Silhouette (ruang sendiri)": silhouette_score(X_condition, labels),
            "Silhouette (ruang bersama)": silhouette_score(X_standard, labels),
        }
    )
result_table = pd.DataFrame(rows).set_index("Kondisi")
print(result_table.round(3).to_string())

## Bagian G: Menggambar empat kondisi

In [ ]:
palette = ["#0072B2", "#E69F00", "#009E73", "#CC79A7", "#D55E00", "#56B4E9"]

fig, axes = plt.subplots(1, 4, figsize=(17, 4.2), sharex=True, sharey=True)
for ax, (name, labels) in zip(axes, labels_by_condition.items()):
    for c in range(5):
        ax.scatter(X_raw[labels == c, 0], X_raw[labels == c, 1], s=14, color=palette[c])
    ari = result_table.loc[name, "ARI vs baseline"]
    ax.set_title(f"{name}\nARI vs baseline = {ari:.3f}", fontsize=9)
    ax.set_xlabel("Pendapatan tahunan (k$)")
axes[0].set_ylabel("Skor belanja (1-100)")
plt.tight_layout()
plt.show()

## Bagian H: Meringkas temuan

In [ ]:
for name in conditions:
    ari = result_table.loc[name, "ARI vs baseline"]
    verdict = "praktis sama dengan baseline" if ari >= 0.95 else "berbeda dari baseline"
    print(f"{name:42s} ARI = {ari:.3f} -> {verdict}")

## Latihan

1. Ulangi bagian kebisingan baseline dengan `n_init=1` (bukan 10) pada seed 0 sampai 9, lalu hitung ARI tiap hasil terhadap baseline. Berapa ARI terkecil dan terbesarnya? Berapa dari sepuluh yang tepat 1,0? Apa artinya untuk variabel terkontrol?
2. Pada kondisi D, jalankan `StandardScaler` setelah mengalikan pendapatan dengan 1000. Berapa ARI terhadap baseline?
3. Ubah K menjadi 4 dan 6 (K berubah, skala tetap `StandardScaler`). Berapa ARI terhadap baseline K = 5? Mengapa ini bukan eksperimen skala lagi?
4. Tulis rancangan eksperimen (pertanyaan, variabel bebas, terikat, terkontrol, baseline) untuk pertanyaan: "Apakah menghapus fitur `Annual Income` mengubah pengelompokan?"
5. Mengapa Silhouette di ruang sendiri bisa menyesatkan saat membandingkan kondisi dengan skala berbeda?
6. Mengapa kita tidak boleh membandingkan nomor kelompok dua kondisi langsung, misalnya `labels_A == labels_B`?

Jawaban lengkap ada di halaman bab.

## Ringkasan

- **Eksperimen** berbeda dari sekadar menjalankan algoritma: ia punya pertanyaan, variabel yang terkontrol, baseline, dan pencatatan.
- Pada clustering yang *unsupervised*, hipotesis dan label benar tidak selalu ada. Yang wajib: **tujuan dan pertanyaan yang jelas**.
- **Variabel bebas** diubah, **terikat** diukur, **terkontrol** ditahan. Ubah satu variabel pada satu waktu.
- **Baseline** adalah pembanding, dan **kebisingannya** perlu diukur. Pada Mall Customers dengan `n_init=10`, ARI antar seed selalu 1,0, tetapi dengan `n_init=1` bervariasi dari 0,692 sampai 1,0.
- Pada Mall Customers, `MinMaxScaler` dan tanpa skala (k$) memberi pengelompokan yang sama dengan `StandardScaler` (ARI 1,000), sedangkan pendapatan dalam dolar tanpa skala mengubahnya (ARI 0,427).
- Ukur semua kondisi di **ruang yang sama**: Silhouette ruang sendiri untuk kondisi D (0,584) menyesatkan, sedangkan di ruang bersama hanya 0,157.
- Batasi kesimpulan sesuai data dan eksperimen. Hindari klaim sebab-akibat.